# Find Landsat matches

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio as rio

from datetime import datetime, timedelta
from pyproj import CRS
from rasterio.warp import transform_bounds
from shapely.geometry import Polygon
from tqdm import tqdm

In [3]:
from sensorsio.mgrs import get_bbox_mgrs_tile, get_mgrs_tiles_from_roi

In [4]:
import sys

sys.path.append("../src")

In [5]:
from etdataset.database import create_ls8_db

### Inputs

In [6]:
if not os.environ.get("DATA_PATH"):
    raise Exception("DATA_PATH not defined")

In [7]:
data_path = os.environ["DATA_PATH"]

In [8]:
if not os.environ.get("METADATA_PATH"):
    raise Exception("METADATA_PATH not defined")

In [9]:
metadata_path = os.environ["METADATA_PATH"]

In [10]:
# Databases paths
ls8_db_path = os.path.join(metadata_path, "landsat_ot_c2_l2_659ed86540bcec65.csv")

In [11]:
# ROI
s2_tile_id = "28PCA"
roi_path = os.path.join(data_path, "zones/Zone_Senegal_Centre.shp")

In [12]:
# Acquisition dates
min_date_str = "2023-02-01"  # Format YYYY-MM-DD
max_date_str = "2023-03-31"  # Format YYYY-MM-DD

In [13]:
cloud_cover = 25
tile_coverage = 33

In [14]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

### Prepare

In [15]:
# Convert dates
min_date = datetime.strptime(min_date_str, "%Y-%m-%d")
max_date = datetime.strptime(max_date_str, "%Y-%m-%d")

In [16]:
s2_tile_bbox = get_bbox_mgrs_tile(s2_tile_id)
s2_tile_crs = 4326

In [17]:
roi = gpd.read_file(roi_path)
roi_crs = roi.crs
roi_bbox = rio.coords.BoundingBox(*roi.bounds.iloc[0].values)
wgs84_bounds = transform_bounds(roi_crs, 4326, *roi_bbox)
roi_poly = Polygon(
    [
        [wgs84_bounds[0], wgs84_bounds[1]],
        [wgs84_bounds[0], wgs84_bounds[3]],
        [wgs84_bounds[2], wgs84_bounds[3]],
        [wgs84_bounds[2], wgs84_bounds[1]],
    ]
)

In [18]:
mgrs_tiles = get_mgrs_tiles_from_roi(roi_bbox, roi_crs)

In [19]:
mgrs_tiles

,Name,overlap_geometry,geometry,overlap_percentage
0,28PBA,"POLYGON ((-16.76482 14.46621, -16.75975 13.800...","POLYGON ((-17.78282 14.45638, -16.76482 14.466...",22.271238
1,28PBB,"POLYGON ((-16.76412 14.37783, -17.10000 14.374...","POLYGON ((-17.79446 15.35961, -16.77221 15.370...",20.551890
2,28PCA,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120620
3,28PCB,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627789
4,28PDA,"POLYGON ((-15.90000 14.47105, -15.90000 13.800...","POLYGON ((-15.92826 14.47100, -14.90943 14.472...",1.789794
5,28PDB,"POLYGON ((-15.92790 14.38258, -15.93054 15.000...","POLYGON ((-15.93215 15.37519, -14.90905 15.377...",1.780280


### Create database

In [20]:
ls8_db = create_ls8_db(
    ls8_db_path, min_date=min_date, max_date=max_date, max_cloud_cover=cloud_cover
)

24-01-23 12:04:31 :: INFO :: Found 35897 Landsat products fullfilling the requirements


In [21]:
ls8_db.tail()

,product_name,date,cloud_cover,geometry
35892,LC09_L2SP_233084_20230331_20230402_02_T1,2023-03-31,0.33,"POLYGON ((-72.30069 -33.54329, -69.82654 -33.5..."
35893,LC09_L2SP_233085_20230331_20230402_02_T1,2023-03-31,0.28,"POLYGON ((-72.72173 -34.97498, -70.20910 -35.0..."
35894,LC09_L2SP_233086_20230331_20230402_02_T1,2023-03-31,0.07,"POLYGON ((-73.15644 -36.40137, -70.59931 -36.4..."
35895,LC09_L2SP_233087_20230331_20230402_02_T1,2023-03-31,3.33,"POLYGON ((-73.74738 -37.85170, -71.03692 -37.7..."
35896,LC09_L2SP_233088_20230331_20230402_02_T1,2023-03-31,5.68,"POLYGON ((-74.21626 -39.27782, -71.45415 -39.2..."


In [22]:
ls8_db = create_ls8_db(
    ls8_db_path,
    min_date=min_date,
    max_date=max_date,
    max_cloud_cover=cloud_cover,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    min_roi_overlap=80,
)

24-01-23 12:04:53 :: INFO :: Found 6 Landsat products fullfilling the requirements


In [23]:
ls8_db

,product_name,date,cloud_cover,overlap_geometry,geometry,overlap_percentage
0,LC09_L2SP_205050_20230207_20230311_02_T1,2023-02-07,19.05,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.78678 15.49572, -15.65362 15.512...",100.0
1,LC09_L2SP_205050_20230223_20230308_02_T1,2023-02-23,0.31,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.74767 15.49621, -15.61446 15.512...",100.0
2,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73373 15.49909, -15.60328 15.515...",100.0
3,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,0.04,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73929 15.49632, -15.60607 15.512...",100.0
4,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,0.66,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73932 15.49902, -15.60607 15.515...",100.0
5,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,0.07,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73370 15.49638, -15.60047 15.512...",100.0


### Write results

In [24]:
ls8_db.to_csv(os.path.join(output_path, "ls8_matches.csv"))